TASK 1.2

In [0]:
spark.sql("DROP TABLE IF EXISTS quiz_attempts_bronze")


In [0]:
from pyspark.sql.functions import *
quiz_path = "/Volumes/edtech_project/edtech_bronze/edtech_project/quiz_attempts/"

# 1. Get last processed timestamp (watermark)
try:
    max_ts = spark.sql("""
        SELECT MAX(submit_ts) as max_ts 
        FROM quiz_attempts_bronze
    """).collect()[0]["max_ts"]
except:
    max_ts = None

# 2. Read CSV
quiz_df = (spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(quiz_path)
)

# 3. Convert timestamp
quiz_df = quiz_df.withColumn("submit_ts", to_timestamp("submit_ts"))

# 4. Incremental filter
if max_ts:
    quiz_df = quiz_df.filter(col("submit_ts") > lit(max_ts))

# 5. Data Quality Checks
quiz_df = (quiz_df
    .filter(col("student_id").isNotNull())
    .filter(col("submit_ts").isNotNull())
)

# 6. Score validation
quiz_df = quiz_df.filter(
    (col("score_obtained") >= 0) &
    (col("score_obtained") <= col("max_score"))
)

# 7. Logical inconsistency flag
quiz_df = quiz_df.withColumn(
    "is_inconsistent",
    when(
        (col("status") == "IN_PROGRESS") & col("submit_ts").isNotNull(),
        lit(True)
    ).otherwise(lit(False))
)

# 8. Metadata
quiz_df = (quiz_df
    .withColumn("_source_file", col("_metadata.file_path"))
    .withColumn("_load_ts", current_timestamp())
    .withColumn("_schema_version", lit("v1"))
)

# 9. Partition column
quiz_df = quiz_df.withColumn("submit_date", to_date("submit_ts"))

# 10. Write
(quiz_df.write
    .format("delta")
    .mode("append")
    .partitionBy("submit_date")
    .saveAsTable("quiz_attempts_bronze")
)

In [0]:

%sql
--display partitions
SHOW PARTITIONS quiz_attempts_bronze;

In [0]:

%sql
-- display the table
SELECT * 
FROM quiz_attempts_bronze
LIMIT 5;


In [0]:
%sql
-- validating the flagged column
SELECT count(*)
FROM quiz_attempts_bronze
WHERE is_inconsistent = true;

In [0]:
%sql
select count(*) from quiz_attempts_bronze;